# Error Analysis &mdash; *Evaluating Commercial AI Chatbots as News Intermediaries*

This notebook does three things:

1. **Re-classify every model–question instance** as correct/incorrect using the canonical answer-extractor from `ReplicateResults.ipynb`.
2. **Regenerate `ablation/incorrect_examples/`** so that there is one `errors_<model>_<region>_part_<N>.txt` file per (model, region) cell. These are the inputs that were fed to the error-taxonomy LLM judges.
3. **Reconstruct the error-taxonomy table** (Table 6) from the three pre-computed judge JSONLs in `ablation/error_analysis/`, using a per-example majority vote with deterministic priority-order tie-break for full-disagreement cases.


## 0 · Setup — paths, models, regions

In [1]:
import json, re, csv
from collections import defaultdict, Counter
from pathlib import Path
import pandas as pd

ROOT      = Path.cwd()
RES       = ROOT / "results"
NEW       = ROOT / "ablation" / "incorrect_examples"
NEW.mkdir(parents=True, exist_ok=True)

REGIONS = ["uscanada", "afrique", "arabic", "hindi", "russian", "turkce"]
REGION_PRETTY = {
    "uscanada": "US & Canada", "afrique": "Afrique", "arabic": "Arabic",
    "hindi": "Hindi", "russian": "Russian", "turkce": "Turkish",
}

MAIN_MODELS = [
    ("gemini-3-flash-preview",      "Gemini 3 Flash"),
    ("gemini-3-pro-preview",        "Gemini 3 Pro"),
    ("grok-4-0709",                 "Grok 4"),
    ("claude-sonnet-4-5",           "Claude 4.5 Sonnet"),
    ("gpt-5-search-api",            "GPT-5"),
    ("gpt-4o-mini-search-preview",  "GPT-4o Mini"),
]
BASELINE_MODELS = [
    ("gemini-3-pro-preview-baseline", "Gemini 3 Pro (no search)"),
    ("grok-4-0709-baseline",          "Grok 4 (no search)"),
    ("claude-sonnet-4-5-baseline",    "Claude 4.5 Sonnet (no search)"),
    ("gpt-5.2-baseline",              "GPT-5 (no search)"),
]
MODEL_DIR_TO_NAME = {**dict(MAIN_MODELS), **dict(BASELINE_MODELS)}

EVAL_DAYS = [f"february_{d}" for d in range(9, 23)]   # Feb 9 - 22, 2026
EXAMPLES_PER_PART = 20
SEPARATOR = "-" * 80

## 1 &middot; Canonical answer extractor

The same regex-based extractor used in `ReplicateResults.ipynb`. Keep this
copy byte-identical to that one; the synthetic tests below fail loudly if the
regex changes.


In [2]:
ANS_TAG_RE     = re.compile(r"<final_answer>(.*?)</final_answer>", re.IGNORECASE | re.DOTALL)
LETTER_PAREN   = re.compile(r"\(\s*([A-Fa-f])\s*\)")
LETTER_STRICT  = re.compile(r"^\s*\(?\s*([A-Fa-f])\s*\)?(?:\s|$|[\.,])")
LETTER_TAIL    = re.compile(r"\(\s*([A-Fa-f])\s*\)\s*$", re.MULTILINE)

def extract_answer(text):
    """Return upper-case letter A-F or None if not parseable.

    Same parsing behavior as the copy in ReplicateResults.ipynb. Inside
    <final_answer>...</final_answer> we prefer an explicit ``(X)`` token;
    fall back to a strict leading bare letter; otherwise return None.
    The earlier permissive regex spuriously extracted letters from
    refusal text such as the F in "Final answer:".
    """
    if not isinstance(text, str):
        return None
    m = ANS_TAG_RE.search(text)
    if m:
        inner = m.group(1)
        paren_matches = LETTER_PAREN.findall(inner)
        if paren_matches:
            return paren_matches[0].upper()
        m2 = LETTER_STRICT.match(inner.lstrip())
        if m2:
            return m2.group(1).upper()
        return None
    matches = LETTER_TAIL.findall(text)
    if matches:
        return matches[-1].upper()
    return None

# Gold-label errata (see ReplicateResults.ipynb for full provenance).
# Multi-letter sets indicate duplicate-option questions where two option
# letters carry identical text and either is acceptable.
GOLD_ERRATA = {
    ("hindi",    "february_21", 19): {"C", "E"},
    ("afrique",  "february_9",  14): {"D", "E"},
    ("arabic",   "february_14", 20): "B",
    ("uscanada", "february_13",  6): "A",
}

def canonical_correct(s, *, region_dir=None, day=None, q_idx=None):
    """Return a single letter A-F, a frozenset of letters (errata case), or None."""
    key = (region_dir, day, q_idx) if region_dir is not None else None
    if key in GOLD_ERRATA:
        v = GOLD_ERRATA[key]
        return frozenset(v) if isinstance(v, (set, frozenset)) else v
    if not isinstance(s, str):
        return None
    m = LETTER_PAREN.search(s) or LETTER_STRICT.match(s.lstrip())
    return m.group(1).upper() if m else None

def is_incorrect(row, *, region_dir=None, day=None, q_idx=None):
    pred    = extract_answer(row.get("model_output", ""))
    correct = canonical_correct(row.get("correct_answer"),
                                region_dir=region_dir, day=day, q_idx=q_idx)
    if correct is None:
        return False, pred, correct
    if isinstance(correct, frozenset):
        return (pred is None or pred not in correct), pred, correct
    return (pred != correct), pred, correct

# `extract_answer` is duplicated verbatim across the three notebooks.
# The synthetic regression tests below break loudly if the regex changes.
assert extract_answer("<final_answer>A</final_answer>") == "A"
assert extract_answer("<final_answer>(B)</final_answer>") == "B"
assert extract_answer("<final_answer>  C  </final_answer>") == "C"
assert extract_answer("<final_answer>(D) An evil operative</final_answer>") == "D"
assert extract_answer("<final_answer>e</final_answer>") == "E"
assert extract_answer("<FINAL_ANSWER>F</FINAL_ANSWER>") == "F"
assert extract_answer("<final_answer>\n(A)\n</final_answer>") == "A"
assert extract_answer("Reasoning... (D)\nFinal: (A)") == "A"
assert extract_answer("I think it's A but maybe B") is None
assert extract_answer(None) is None
assert extract_answer("") is None
assert extract_answer("<final_answer></final_answer>") is None
assert extract_answer("<final_answer>G</final_answer>") is None
# v2 strict regression tests: refusal/preamble strings must NOT extract a letter.
assert extract_answer("<final_answer>Final answer: (B)</final_answer>") == "B"
assert extract_answer("<final_answer>The answer is C</final_answer>") is None
assert extract_answer("<final_answer>I cannot answer.</final_answer>") is None
print("extract_answer parity-lock tests pass")


extract_answer parity-lock tests pass


## 2 · Build the master error table

Loads all 6 main models × 6 regions × 14 days plus the 4 baseline (no-search) models × 6 regions and tags every question instance with `is_incorrect` + the canonically-extracted `pred`.

In [3]:
def load_records(model_dir, region_dir):
    rows = []
    for day in EVAL_DAYS:
        path = RES / region_dir / model_dir / f"{day}_outputs.jsonl"
        if not path.exists():
            continue
        with open(path) as f:
            day_rows = [json.loads(line) for line in f if line.strip()]
        for q_idx, r in enumerate(day_rows):
            wrong, pred, correct = is_incorrect(
                r, region_dir=region_dir, day=day, q_idx=q_idx,
            )
            rows.append({
                "model_dir": model_dir,
                "model":     MODEL_DIR_TO_NAME.get(model_dir, model_dir),
                "region_dir": region_dir,
                "region":    REGION_PRETTY[region_dir],
                "day":       day,
                "q_idx":     q_idx,
                "correct":   correct,
                "pred":      pred,
                "is_incorrect": wrong,
                "format_failure": (pred is None) and wrong,
                "baseline":  model_dir.endswith("-baseline"),
            })
    return rows

all_rows = []
for model, _ in MAIN_MODELS + BASELINE_MODELS:
    for region in REGIONS:
        all_rows.extend(load_records(model, region))

df = pd.DataFrame(all_rows)
print(f"loaded {len(df):,} model-question instances")
print(f"  main models:     {(~df['baseline']).sum():>6,d}")
print(f"  baseline models: {df['baseline'].sum():>6,d}")

loaded 20,508 model-question instances
  main models:     12,600
  baseline models:  7,908


## 3 &middot; Sanity check on the master error table

Per-model accuracy and error counts for the main (search-enabled) models.


In [4]:
main = df[~df["baseline"]]
acc_by_model  = (1 - main.groupby("model")["is_incorrect"].mean()) * 100
errs_by_model = main.groupby("model")["is_incorrect"].sum().astype(int)
table = pd.concat([acc_by_model.round(2).rename("acc_%"),
                   errs_by_model.rename("n_errors")], axis=1)
table = table.reindex([n for _, n in MAIN_MODELS])
print(table)
print(f"\nTotal main-model errors: {errs_by_model.sum():,}")

                   acc_%  n_errors
model                             
Gemini 3 Flash     95.57        93
Gemini 3 Pro       93.71       132
Grok 4             95.05       104
Claude 4.5 Sonnet  90.43       201
GPT-5              84.95       316
GPT-4o Mini        69.00       651

Total main-model errors: 1,497


## 4 &middot; Regenerate `ablation/incorrect_examples/`

One file per (model, region) cell, chunked into parts of `EXAMPLES_PER_PART = 20`.
Each part file contains the question, model output, correct answer, and predicted
answer for a chunk of misclassified examples; these are the inputs the
error-taxonomy LLM judges saw.


In [5]:
def _str(v):
    return v.rstrip() if isinstance(v, str) else ""

# The JSONL `input` field begins with a `GENERAL INSTRUCTIONS:` preamble
# (the same prompt for every question), separated from the actual question
# by a line of `=` signs. We strip the preamble so each part file contains
# only the question + model output.
DIVIDER_RE = re.compile(r"={5,}\s*")
def strip_instructions(input_text):
    if not isinstance(input_text, str):
        return ""
    parts = DIVIDER_RE.split(input_text)
    # Question is everything after the LAST divider; fall back to the full
    # input if no divider is present (e.g., a baseline run that omits the
    # preamble entirely).
    return parts[-1].strip() if len(parts) > 1 else input_text.strip()

def format_example(idx, region_dir, day, model_dir, row, pred, correct):
    pred_str = f"({pred})" if pred else ""
    return (
        f"METADATA [{idx}]: index: {idx}, language: {region_dir}, date: {day}, model: {model_dir}\n"
        "ORIGINAL EXPLANATION (NOT SEEN OR USED BY MODEL; USED BY HUMANS TO GENERATE QUESTION AND CONFIRM CORRECT ANSWER):\n"
        f"{_str(row.get('explanation'))}\n"
        "INPUT:\n\n\n"
        f"{strip_instructions(row.get('input'))}\n"
        "MODEL OUTPUT:\n"
        f"{_str(row.get('model_output'))}\n"
        "CORRECT ANSWER:\n"
        f"({correct})\n"
        "PREDICTED ANSWER:\n"
        f"{pred_str}\n"
        f"{SEPARATOR}\n"
    )

def write_cell(model_dir, region_dir):
    """Re-read JSONL (we need the original row contents, not just the flag),
    apply the extractor, and write the part files."""
    errs = []
    for day in EVAL_DAYS:
        path = RES / region_dir / model_dir / f"{day}_outputs.jsonl"
        if not path.exists():
            continue
        with open(path) as f:
            day_rows = [json.loads(line) for line in f if line.strip()]
        for q_idx, r in enumerate(day_rows):
            wrong, pred, correct = is_incorrect(
                r, region_dir=region_dir, day=day, q_idx=q_idx,
            )
            if wrong:
                errs.append((q_idx, day, r, pred, correct))
    n_parts = 0
    for start in range(0, len(errs), EXAMPLES_PER_PART):
        chunk = errs[start : start + EXAMPLES_PER_PART]
        out = NEW / f"errors_{model_dir}_{region_dir}_part_{start // EXAMPLES_PER_PART}.txt"
        with open(out, "w") as f:
            for q_idx, day, row, pred, correct in chunk:
                f.write(format_example(q_idx, region_dir, day, model_dir, row, pred, correct))
        n_parts += 1
    return len(errs), n_parts

# Wipe and rewrite so the cell is idempotent.
for f in NEW.glob("errors_*.txt"):
    f.unlink()

manifest = []
for model, _ in MAIN_MODELS + BASELINE_MODELS:
    for region in REGIONS:
        if not (RES / region / model).is_dir():
            continue
        n, p = write_cell(model, region)
        manifest.append({"model_dir": model, "region_dir": region,
                         "n_errors": n, "n_part_files": p,
                         "baseline": model.endswith("-baseline")})

manifest_df = pd.DataFrame(manifest)
print(f"wrote {len(manifest_df)} cells x {EXAMPLES_PER_PART}-example part files into {NEW}")
manifest_df

wrote 60 cells x 20-example part files into <repo>/ablation/incorrect_examples


,model_dir,region_dir,n_errors,n_part_files,baseline
0,gemini-3-flash-preview,uscanada,12,1,False
1,gemini-3-flash-preview,afrique,18,1,False
2,gemini-3-flash-preview,arabic,8,1,False
3,gemini-3-flash-preview,hindi,35,2,False
4,gemini-3-flash-preview,russian,7,1,False
5,gemini-3-flash-preview,turkce,13,1,False
6,gemini-3-pro-preview,uscanada,14,1,False
7,gemini-3-pro-preview,afrique,20,1,False
8,gemini-3-pro-preview,arabic,32,2,False
9,gemini-3-pro-preview,hindi,42,3,False


## 5 &middot; Error-taxonomy classification (3-judge majority vote)

The part files generated above were classified by three independent LLM judges
(GPT-5.2, Claude Sonnet 4.6, Gemini 3 Pro). Their output JSONLs are stored in
`ablation/error_analysis/`.

The cells below load the three judge outputs, take the per-example majority
vote, and apply a **deterministic priority-order tie-break** for full-disagreement
(1, 1, 1) cases &mdash; using the priority order documented in the classifier
prompt (Retrieval Failure → Event/Entity Misbinding → Source Divergence
→ Temporal Misalignment → Answer Formatting → Inference Override
→ Qualifier Binding → Reading Comprehension) &mdash; to reconstruct
the error-taxonomy table.


In [6]:
# Load each judge's per-example labels.
JUDGE_FILES = {
    "gpt52":  ROOT / "ablation/error_analysis/error_taxonomy_outputs_gpt-5.2.jsonl",
    "claude": ROOT / "ablation/error_analysis/error_taxonomy_outputs_claude.jsonl",
    "gemini": ROOT / "ablation/error_analysis/error_taxonomy_outputs_gemini-3-pro-preview.jsonl",
}

CANONICAL = [
    "Retrieval Failure with Forced Guessing",
    "Event / Entity Misbinding",
    "Source Divergence Under Real-World Report Conflict",
    "Temporal Misalignment / Chronology Tracking Failure",
    "Answer Formatting / Option Mapping Failure",
    "Inference Override / Over-Reasoning Instead of Extraction",
    "Qualifier Binding Error (Unit/Currency/Year/Version)",
    "Reading Comprehension / Option Interpretation Failure",
]
TIEBREAK_RANK = {c: i for i, c in enumerate(CANONICAL)}   # priority 1 → rank 0
CANON_SET = set(CANONICAL)

EX_RE = re.compile(
    r"index:\s*(?P<idx>\d+).*?language:\s*(?P<region>[a-z]+).*?"
    r"date:\s*(?P<day>february_\d+).*?model:\s*(?P<model>\S+)",
    re.IGNORECASE,
)

def parse_judge(path):
    out = {}
    if not path.is_file():
        return out
    for line in path.read_text().splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)
        body = rec.get("model_output", "") or ""
        body = re.sub(r"^```(?:json|jsonl)?\s*", "", body.strip())
        body = re.sub(r"\s*```$", "", body)
        for sub in body.splitlines():
            sub = sub.strip().rstrip(",")
            if not sub.startswith("{"):
                continue
            try:
                obj = json.loads(sub)
            except json.JSONDecodeError:
                continue
            ex_str = obj.get("example", "")
            cat = (obj.get("error_category") or "").strip()
            m = EX_RE.search(ex_str)
            if not m or cat not in CANON_SET:
                continue
            out[(m["model"], m["region"], m["day"], int(m["idx"]))] = cat
    return out

judge_labels = {j: parse_judge(p) for j, p in JUDGE_FILES.items()}
print({j: len(d) for j, d in judge_labels.items()})


{'gpt52': 1497, 'claude': 1497, 'gemini': 1497}


In [7]:
# Per-example majority vote with deterministic priority tie-break for 1,1,1 splits.
all_keys = sorted(set().union(*judge_labels.values()))
final_counts = Counter()
unanimous = two_of_three = tiebroken = 0
final_rows = []

for key in all_keys:
    votes = [judge_labels[j].get(key) for j in JUDGE_FILES]
    votes_clean = [v for v in votes if v]
    if not votes_clean:
        continue
    c = Counter(votes_clean)
    top, top_n = c.most_common(1)[0]
    if top_n == 3:
        chosen, kind = top, "unanimous"; unanimous += 1
    elif top_n == 2:
        chosen, kind = top, "majority";  two_of_three += 1
    else:
        chosen = min(votes_clean, key=lambda v: TIEBREAK_RANK[v])
        kind   = "tiebroken"; tiebroken += 1
    final_counts[chosen] += 1
    final_rows.append({
        "model_dir": key[0], "region_dir": key[1], "day": key[2], "q_idx": key[3],
        "gpt52": votes[0] or "", "claude": votes[1] or "", "gemini": votes[2] or "",
        "vote_kind": kind, "final_category": chosen,
    })

print(f"unanimous: {unanimous}   2-of-3: {two_of_three}   1,1,1 tiebroken: {tiebroken}   total: {len(final_rows)}")


unanimous: 816   2-of-3: 540   1,1,1 tiebroken: 141   total: 1497


In [8]:
total = sum(final_counts.values())
table6 = pd.DataFrame([
    {"category": cat, "n": final_counts.get(cat, 0),
     "pct":   round(final_counts.get(cat, 0) / total * 100, 1)}
    for cat in CANONICAL
]).sort_values("n", ascending=False).reset_index(drop=True)
table6.loc[len(table6)] = ["TOTAL", total, 100.0]
table6


,category,n,pct
0,Retrieval Failure with Forced Guessing,581,38.8
1,Source Divergence Under Real-World Report Conf...,490,32.7
2,Answer Formatting / Option Mapping Failure,137,9.2
3,Inference Override / Over-Reasoning Instead of...,123,8.2
4,Event / Entity Misbinding,67,4.5
5,Reading Comprehension / Option Interpretation ...,49,3.3
6,Temporal Misalignment / Chronology Tracking Fa...,27,1.8
7,Qualifier Binding Error (Unit/Currency/Year/Ve...,23,1.5
8,TOTAL,1497,100.0


In [9]:
# Persist artifacts for downstream use.
ANA = ROOT / "ablation" / "error_analysis"
labels_csv = ANA / "_per_example_labels.csv"
with open(labels_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(final_rows[0].keys()))
    w.writeheader()
    w.writerows(final_rows)

table6_csv = ANA / "_table6_majority_vote.csv"
table6.iloc[:-1].to_csv(table6_csv, index=False)

print(f"wrote {labels_csv}  ({len(final_rows)} rows)")
print(f"wrote {table6_csv}")


wrote <repo>/ablation/error_analysis/_per_example_labels.csv  (1497 rows)
wrote <repo>/ablation/error_analysis/_table6_majority_vote.csv
